## A3 - Calibration and Position Estimation
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 13th, 2026

About: In this notebook you'll calibrate your robot's motion so that you can plan a trajectory for your robot.

## Our robots do not have a Global Positioning System, that can be a problem.
If we don't know the position of the robot relative to other objects, it can easily colide with them.  
In this notebook we'll learn how to convert our robot commands to actual inches and feet measurements in the real world.  
So that we can use our robots in real and interesting environments we need to measure our motion in units of meters.

### Calibration
**Calibration** - comparing a device's measurement or action against a standard reference measurement.  
We are now going to calibrate our robot or measure it's driving distance with a ruler.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY A3.1: Motion Measurement</span>


In this first activity, we are going to measure how far our robots travel over 0.5, 1, 1.5, 2, and 2.5 seconds.  
We can then plot that data to come up with a function that helps us to drive a specific distance *on command*!

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# This cell sets up the commands we'll use for the rest of this notebook

#import the Sphero RVR Tank Robot Software Development Kit
from sphero_sdk import RawMotorModesEnum
from sphero_sdk import DriveFlagsBitmask
import time

# Import our shared robot connection helpers
from robot_utils import get_rvr, close_if_exists

# Prepare the interface for the robot using our shared robot_utils helper.
# get_rvr() connects to the robot and wakes it up, and safely reuses an
# existing connection if you re-run this cell.
rvr = get_rvr()

# Our function for driving the robot with speed, heading, and drive time
def drive_robot(speed_rate, desired_heading, drive_time ): 
    # Reset the heading for this program
    rvr.drive_control.reset_heading()
    
    rvr.drive_control.drive_forward_seconds(
        speed=speed_rate, # Now using the user supplied input argument
        heading=desired_heading,  # Now using the user supplied input argument
        time_to_drive=drive_time # Now using the user supplied input argument
    )

In [ ]:
#### ------> ACTIVITY A3.1: Motion Measurement <-----#####
# Command your robot to move at a constant speed for differnt lengths of time, then measure how far your robot traveled.

##### INSTRUCTIONS: #####
# 1. Set your constant speed rate below, don't change this afterwards. 
# 2. Set your robot at the designed starting line then command it to move forward for trials of 0.5, 1, 1.5, 2, and 2.5 seconds. 
# 3. After each trial record the distance the rover moved in units of meters. Record the distances below.

robot_speed = 100 # 100 is a good start, but you're free to change it, after changing it stick with your choice.

drive_robot(robot_speed, 0, 0.5)

In [ ]:
#### ------> Record Results Here <-----#####
# After each trial, type the measured distance (in meters) to the right of the '=' sign.

trial_1 =     # (meters) resulting from 0.5 seconds of drive
trial_2 =     # (meters) resulting from 1.0 seconds of drive
trial_3 =     # (meters) resulting from 1.5 seconds of drive
trial_4 =     # (meters) resulting from 2.0 seconds of drive
trial_5 =     # (meters) resulting from 2.5 seconds of drive

# Now we gather our measurements into two matching lists so we can plot and analyze them.
# 'times' are the drive times we commanded; 'distances' are the distances you measured.
# IMPORTANT: the order must match! trial_1 goes with 0.5s, trial_2 with 1.0s, and so on.
times = [0.5, 1.0, 1.5, 2.0, 2.5]
distances = [trial_1, trial_2, trial_3, trial_4, trial_5]

print("Drive times (s):", times)
print("Measured distances (m):", distances)

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY A3.2: Plot Results and Curve Fitting</span>


Now that we have our measurements, let's **plot** them. On the horizontal axis (x) we'll put the distance the robot traveled, and on the vertical axis (y) we'll put the drive time.

Why distance on x and time on y? Because in our final challenge we'll be *given a distance* (the goal) and we'll need to figure out the *drive time* to reach it. So we want a model that takes a distance in and gives a time out.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# This cell plots your measurements so you can see the relationship between distance and time.

import matplotlib.pyplot as plt

# Plot distance (x) against drive time (y) as individual data points
plt.scatter(distances, times, color="blue", label="Your measurements")
plt.xlabel("Distance traveled (meters)")
plt.ylabel("Drive time (seconds)")
plt.title("Drive Time vs. Distance")
plt.grid(True)
plt.legend()
plt.show()

### Fitting a line to our data
Look at your plotted points. Do they roughly form a straight line? For a robot driving at a constant speed, they should be close!

Real measurements are never perfect — the points will be a little scattered. So instead of connecting the dots, we find the single straight line that best fits **all** of the points at once. This is called a **line of best fit**, and the math behind it is called **linear regression**.

A straight line follows the equation:

$$ time = m \times distance + b $$

where **m** is the slope (how much the time changes for each extra meter) and **b** is the y-intercept (where the line crosses the vertical axis). We'll let Python find the best **m** and **b** for us.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# This cell calculates the line of best fit for your data using linear regression.

import numpy as np

# np.polyfit finds the slope (m) and intercept (b) of the best-fit line.
# We fit with distance as the input (x) and time as the output (y).
# The '1' means we want a 1st-degree polynomial, which is a straight line.
m, b = np.polyfit(distances, times, 1)

print("Slope (m):", m)
print("Intercept (b):", b)
print()
print("Your time-to-distance model is:")
print("    time = " + str(round(m, 4)) + " * distance + " + str(round(b, 4)))

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# This cell re-plots your data WITH the best-fit line drawn through it.

import numpy as np
import matplotlib.pyplot as plt

# Create a smooth set of distance values spanning your data, to draw the line
distance_line = np.linspace(min(distances), max(distances), 100)
time_line = m * distance_line + b   # using the m and b from the previous cell

plt.scatter(distances, times, color="blue", label="Your measurements")
plt.plot(distance_line, time_line, color="red", label="Line of best fit")
plt.xlabel("Distance traveled (meters)")
plt.ylabel("Drive time (seconds)")
plt.title("Drive Time vs. Distance with Best-Fit Line")
plt.grid(True)
plt.legend()
plt.show()

### Using our model: a `time_for_distance` function
Our model lets us answer the key question: *"If I want to drive a certain distance, how long should I drive?"*

Let's package the equation into a function so it's easy to use over and over.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# This function uses your model (m and b) to convert a goal distance into a drive time.

def time_for_distance(goal_distance):
    # Apply the line-of-best-fit equation: time = m * distance + b
    drive_time = m * goal_distance + b
    return drive_time

# Try it out! How long should we drive to travel 2 meters?
print("To drive 2 meters, command a drive time of:", round(time_for_distance(2), 3), "seconds")

<span style="color: green; font-size: 55px; font-style: italic;">Student Discussion Time</span>


## **STUDENT QUESTIONS**:
Talk these over with your group before moving on to the challenge.

 + **Reading the model:** In your equation `time = m * distance + b`, what does the slope **m** tell you about your robot? What are its units? (Hint: it's seconds per meter.)

 + **The intercept:** Your line probably doesn't pass exactly through (0, 0). What could cause the robot to need a little time **b** even for a distance of zero? Think about what the robot does at the very start of a drive.

 + **Interpolation vs. extrapolation:** You measured distances up to about 2.5 seconds of driving. Do you trust your model for a goal that's *farther* than anything you measured? Why might the model become less accurate out there?

 + **Sources of error:** Name two reasons why two robots given the *exact same command* might travel *different* distances. (Think about batteries, wheels, and the floor.)

 + **Using the function:** Explain in your own words what `time_for_distance(4)` returns and how you would use that number to command your robot.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY A3.3: The Precision Driving Challenge</span>


### The Challenge
Now we put your model to the test! Staff have set up **three goals**, at **3 meters, 4 meters, and 5 meters** from the starting line.

**How it works:**
 + For each goal, you get **two attempts**. Use your `time_for_distance()` function to calculate the drive time, then command your robot to drive that long.
 + After each attempt, staff will **measure the distance between your robot and the goal**. Smaller is better — a perfect run is 0 meters of error.
 + Only your **better (shorter) attempt** counts for each goal.
 + Your score is the **sum of your three best attempts** (the 3m, 4m, and 5m errors added together).

### The lowest total error wins the challenge! 🏆

**Strategy tips:**
 + Use the **same `robot_speed`** you calibrated with. If you change the speed, your model is no longer valid!
 + Make sure you re-ran the cells above so `m`, `b`, and `time_for_distance()` are all defined.
 + Line your robot up carefully at the start of each attempt — a crooked start costs you distance.

In [ ]:
#### ------> ACTIVITY A3.3: The Precision Driving Challenge <-----#####
# Use your model to drive to each goal. Fill in the goal distance, run, measure, and try to improve on attempt 2.

##### INSTRUCTIONS: #####
# 1. Set 'goal' to the distance of the goal you're aiming for (3, 4, or 5 meters).
# 2. Run the cell. It calculates the drive time from your model and commands the robot.
# 3. Have staff measure how close you got. Then take your second attempt.

# ---- Set your goal distance here ----
goal =     #<<<<<< replace this with the goal distance in meters (3, 4, or 5) >>>>>>

# Calculate the drive time from your model
commanded_time = time_for_distance(goal)
print("Goal distance:", goal, "meters")
print("Commanded drive time:", round(commanded_time, 3), "seconds")

# Command the robot to drive using your calibrated speed and the computed time.
# NOTE: robot_speed was set back in Activity A3.1 -- use that SAME value.
drive_robot(robot_speed, 0, commanded_time)

### Wrapping Up
Before you move on to the next notebook, run the cell below to release the robot's connection. This keeps things clean for whichever notebook you open next.

In [ ]:
#### ------> RUN THIS CELL WHEN YOU'RE DONE WITH THIS NOTEBOOK <-----#####
close_if_exists()
print("Robot connection closed. Safe to move on to the next notebook!")